# 01. CV 구종 그룹 분류기 — 학습·검증

`scripts/build_pitch_group_dataset.py`가 생성한 `output/pitch_type_cv/dataset.csv`로
궤적 기반 3그룹(FASTBALL/BREAKING/OFFSPEED) 분류기를 학습하고, 홀드아웃 경기로 검증한다.

성공 기준: 3그룹 랜덤 베이스라인(33%)을 유의미하게 상회하는지 확인 (완벽한 정확도가 목표가 아님).

In [ ]:
import os
import sys

ROOT = os.path.dirname(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from pitch_type_cv.group_classifier import predict_group, save_classifier, train_classifier
from pitch_type_cv.trajectory_features import FEATURE_COLUMNS

DATASET_PATH = os.path.join(ROOT, "output", "pitch_type_cv", "dataset.csv")
OUT_DIR = os.path.join(ROOT, "output", "pitch_type_cv")

df = pd.read_csv(DATASET_PATH)
print(f"전체 샘플 수: {len(df)}")
df["group"].value_counts()

## 홀드아웃 경기 분리

`game_pk` 중 하나를 통째로 홀드아웃으로 뺀다 (경기 내 데이터 누수를 막기 위해 투구 단위가 아닌
경기 단위로 분리).

In [ ]:
game_pks = df["game_pk"].unique()
holdout_game_pk = game_pks[-1]
print(f"홀드아웃 경기: {holdout_game_pk} (전체 {len(game_pks)}경기 중)")

train_df = df[df["game_pk"] != holdout_game_pk].reset_index(drop=True)
holdout_df = df[df["game_pk"] == holdout_game_pk].reset_index(drop=True)

print(f"학습 샘플: {len(train_df)}  홀드아웃 샘플: {len(holdout_df)}")

In [ ]:
model = train_classifier(train_df[FEATURE_COLUMNS + ["group"]], train_df["group"].tolist())

y_true = holdout_df["group"].tolist()
y_pred = [predict_group(model, row.to_dict())[0] for _, row in holdout_df[FEATURE_COLUMNS].iterrows()]

accuracy = accuracy_score(y_true, y_pred)
baseline = 1 / 3
print(f"홀드아웃 정확도: {accuracy:.3f}  (랜덤 베이스라인: {baseline:.3f})")
print(classification_report(y_true, y_pred))

## 혼동행렬 & 그룹별 정확도 시각화

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
labels = ["FASTBALL", "BREAKING", "OFFSPEED"]
cm = confusion_matrix(y_true, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap="Blues", ax=ax)
ax.set_xlabel("예측")
ax.set_ylabel("실제")
ax.set_title(f"홀드아웃 혼동행렬 (game_pk={holdout_game_pk})")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

In [ ]:
per_group_acc = (
    pd.DataFrame({"true": y_true, "pred": y_pred})
    .assign(correct=lambda d: d["true"] == d["pred"])
    .groupby("true")["correct"].mean()
    .reindex(labels)
)

fig, ax = plt.subplots(figsize=(5, 4))
per_group_acc.plot(kind="bar", ax=ax, color="#3b82f6")
ax.axhline(baseline, color="red", linestyle="--", label="랜덤 베이스라인 (33%)")
ax.set_ylabel("정확도")
ax.set_title("그룹별 정확도")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "group_accuracy.png"), dpi=150)
plt.show()

In [ ]:
model_path = os.path.join(OUT_DIR, "group_classifier.pkl")
save_classifier(model, model_path)
print(f"모델 저장 완료: {model_path}")

## 해석

- `accuracy`가 33%(랜덤 베이스라인)를 유의미하게 상회하면 궤적 기반 분류 방향이 통한다는 신호.
- 파일럿 규모(5-10경기)이므로 특정 그룹 샘플 부족으로 정확도가 불안정할 수 있음 — 그룹별 정확도
  그래프에서 표본 수가 적은 그룹은 참고용으로만 본다.
- 다음 단계(별도 스펙): 데이터 규모 확대, 다음 구종 예측 파이프라인과의 연결, Streamlit 통합.